In [17]:
import pandas as pd
import numpy as np

# 1. Chargement du jeu d'entraînement
path = "../data/UNSW_NB15_training-set.parquet"
df = pd.read_parquet(path)

initial_rows = df.shape[0]
initial_cols = df.shape[1]

In [11]:
# 2. Suppression des colonnes inutiles ou sources de fuite de données (Data Leakage)
# - 'id' : identifiant séquentiel sans valeur prédictive (si présent)
# - 'attack_cat' : cible multi-classes qui révélerait directement la réponse binaire
cols_to_drop = ['id', 'attack_cat']
existing_drop_cols = [col for col in cols_to_drop if col in df.columns]

df_clean = df.drop(columns=existing_drop_cols)
print(f"Colonnes supprimées : {existing_drop_cols}")

Colonnes supprimées : ['attack_cat']


In [12]:
# 3. Traitement des valeurs infinies (inf / -inf) et manquantes (NaN)
num_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[num_cols] = df_clean[num_cols].replace([np.inf, -np.inf], np.nan)

nan_count = df_clean.isnull().sum().sum()
if nan_count > 0:
    df_clean = df_clean.dropna()
    print(f"Lignes contenant des NaN/Inf supprimées : {nan_count}")
else:
    print("Aucune valeur NaN ou infinie détectée.")

Aucune valeur NaN ou infinie détectée.


In [13]:
# 4. Suppression des doublons exacts
duplicates_count = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Doublons exacts supprimés : {duplicates_count:,}")

Doublons exacts supprimés : 82,984


In [19]:
# 5. Bilan du nettoyage
print("\n=== BILAN DU NETTOYAGE ===")
print(f"Dimensions initiales : {initial_rows:,} lignes x {initial_cols} colonnes")
print(f"Dimensions finales   : {df_clean.shape[0]:,} lignes x {df_clean.shape[1]} colonnes")
print(f"Rétention des données : {(df_clean.shape[0] / initial_rows * 100):.2f}%")


=== BILAN DU NETTOYAGE ===
Dimensions initiales : 175,341 lignes x 36 colonnes
Dimensions finales   : 92,357 lignes x 35 colonnes
Rétention des données : 52.67%
